Overall process
1. Create github repository > .gitignore | LICENSE | README.md
2. Cloned the repo
3. Updated README.md
4. Created initial folder structure
5. Create virtual environment
6. Install dependencies
7. Design overall workflow (in experimentations)
    - setup and config
    - create a client
    - create prompt template
    - handle input (msgs - scam or not)
    - generate response with llm
    - handle output (parsing)
8. Convert it into project files

In [4]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv
load_dotenv()


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL_NAME= "gemini-2.5-flash-lite"

In [6]:
# 3. create prompt template
PROMPT_TEMPLATE = """
You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {{
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }}
```
User Message:
{}
"""

In [10]:
# user_input = input("Enter the text message to analyze: ")
user_input = "Your package delivery failed. Please update your address at [www.bestwebsite.com] to avoid return to sender."
# print(user_input)

full_prompt = PROMPT_TEMPLATE.format(user_input)
# print(full_prompt)

In [11]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [12]:
response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=full_prompt
)

In [13]:
print(response.text)

1. **Thought**: The message creates a sense of urgency ("delivery failed", "avoid return to sender") and prompts the user to click on a link to "update address". This is a common tactic used in phishing scams, where attackers try to trick users into revealing personal information or downloading malware. The URL provided, "www.bestwebsite.com", is generic and could easily be a spoofed website designed to look legitimate. Legitimate delivery services usually provide tracking numbers and have more specific website URLs.

2. **Action**: Based on the urgency, the generic and potentially fake URL, and the common scam pattern of "failed delivery" notifications, this message is likely a scam.

3. **Final Answer**:
```json
{
  "label": "Scam",
  "reasoning": "The message uses urgency ('package delivery failed', 'avoid return to sender') to pressure the recipient into action. It directs the user to a generic and potentially malicious link ('www.bestwebsite.com') to 'update their address'. This i

In [15]:
import json
import re


def extract_json(text):
    """
    Finds the first '{' and the last '}' in a string 
    and parses it as JSON.
    """
    try:
        # Regex to find the JSON block inside the text
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"JSON Parsing Error: {e}")
    
    # Fallback if parsing fails
    return {
        "label": "Uncertain", 
        "reasoning": "Could not parse AI response",
        "intent": "Unknown",
        "risk_factors": ["Parsing Error"]
    }

parsed_output = extract_json(response.text)
print("="*40)
print(parsed_output)

{'label': 'Scam', 'reasoning': "The message uses urgency ('package delivery failed', 'avoid return to sender') to pressure the recipient into action. It directs the user to a generic and potentially malicious link ('www.bestwebsite.com') to 'update their address'. This is a common phishing tactic to steal personal information or deploy malware. Legitimate delivery companies typically have more specific URLs and provide tracking information.", 'intent': 'To phish for personal information (like address, login credentials, or payment details) or to trick the user into downloading malware by impersonating a delivery service.', 'risk_factors': ['Urgency and threat of negative consequence (package return)', 'Generic and potentially fake URL for address update', 'Impersonation of a delivery service', 'Request for personal information via a link']}


In [ ]:
type(parsed_output)